<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/GameTheoryChpater6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow
!pip install keras

In [ ]:
def build_generator(z_size, g_hidden_size, g_output_size):
    model = Sequential([
        Input(shape=(z_size,)),
        Dense(g_hidden_size),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(g_hidden_size * 2),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(g_hidden_size * 4),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(g_output_size, activation='tanh')
    ])
    return model

In [ ]:
def build_discriminator(input_size, d_hidden_size, d_output_size):
    model = Sequential([
        Input(shape=(input_size,)),
        Dense(d_hidden_size * 4),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(d_hidden_size * 2),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(d_hidden_size),
        LeakyReLU(negative_slope=0.2),
        Dropout(0.3),
        Dense(d_output_size, activation='sigmoid')
    ])
    return model


In [ ]:
def __init__(self, real_images, batch_size, z_size, generator, **kwargs):
        super().__init__(**kwargs)
        self.real_images = real_images
        self.batch_size = batch_size
        self.z_size = z_size
        self.generator = generator


def __len__(self):
        return len(self.real_images) // self.batch_size


def __getitem__(self, index):
        noise = np.random.normal(0, 1, (self.batch_size, self.z_size))
        real_imgs = self.real_images[index * self.batch_size:(index + 1) * self.batch_size]
        fake_imgs = self.generator.predict(noise, verbose=0)
        x = np.concatenate([real_imgs, fake_imgs])
        real_labels = np.ones((self.batch_size, 1))
        fake_labels = np.zeros((self.batch_size, 1))
        y = np.concatenate([real_labels, fake_labels])
        return x, y


In [ ]:
class LossHistory(Callback):
  def on_train_begin(self, logs=None):
    self.losses = {'d_loss': [], 'g_loss': []}

  def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            self.losses['d_loss'].append(logs.get('loss'))
            self.losses['g_loss'].append(logs.get('g_loss'))

In [ ]:
class TrainGeneratorCallback(Callback):
    def __init__(self, generator, combined, z_size, batch_size):
        super().__init__()
        self.generator = generator
        self.combined = combined
        self.z_size = z_size
        self.batch_size = batch_size


    def on_epoch_end(self, epoch, logs=None):
        noise = np.random.normal(0, 1, (self.batch_size * 2, self.z_size))
        misleading_targets = np.ones((self.batch_size * 2, 1))
        history = self.combined.fit(noise, misleading_targets, batch_size=self.batch_size, epochs=1, verbose=0)
        g_loss = history.history['loss'][-1]
        if logs is None:
            logs = {}
        logs['g_loss'] = g_loss
